In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.append('.')

import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer, train_bpe
from storybot.utils import get_device, generate
import pickle

In [2]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)
    
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)
    return 0.0

def get_batch(data, context_len, batch_size, device, random=True, offset=0):
    if random:
        ix = torch.randint(len(data) - context_len, (batch_size, ))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)
        ix = ix[ix + context_len + 1 <= len(data)]
        if len(ix) == 0:
            return None, None

    x = torch.stack([torch.from_numpy(data[i:i + context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

def evaluate(model, val_data, context_len, batch_size, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc='Validation'):
            offset = batch_idx * batch_size * context_len
            x, y = get_batch(val_data, context_len, batch_size, device,
                             random=False, offset=offset)
            if x is None:
                break
            with autocast(device_type=device.type, dtype=torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       y.view(-1), reduction='sum')
            total_loss += loss.item()
            total_tokens += y.numel()
    model.train()
    return total_loss / total_tokens

In [3]:
device = get_device()
print(device)

vocab_size = 10000
file_path = 'storybot/tiny_stories_train.txt'

merge_rules = train_bpe(file_path, vocab_size, num_processes = 8)
with open('storybot/merge_rules.pkl', 'wb') as f:
    pickle.dump(merge_rules, f)

cuda


Training BPE: 100%|██████████| 9743/9743 [00:35<00:00, 273.98it/s]


In [4]:
merge_rules_path = 'storybot/merge_rules.pkl'
tokenizer = BPETokenizer.load_from(merge_rules_path)

tokenizer.encode_file(
    'storybot/tiny_stories_train.txt',
    'storybot/tiny_stories_train.bin', num_processes = 8
)

tokenizer.encode_file(
    'storybot/tiny_stories_valid.txt',
    'storybot/tiny_stories_valid.bin', num_processes = 8
)

Encoding chunks: 100%|██████████| 64/64 [00:04<00:00, 14.57it/s]


5465873

In [5]:
device = get_device()
print(device)

data_path = 'storybot/tiny_stories_train.bin'
valid_data_path = 'storybot/tiny_stories_valid.bin'
tokenizer_path = 'storybot/merge_rules.pkl'
model_save_path = 'storybot/model_pretrain.pt'

context_len = 256
vocab_size = 10000
batch_size = 32
learning_rate = 0.001
warmup_iters = 200
# max_iters = 40000
max_iters = 10000
embed_dim = 512
n_head = 16
n_layer = 4
ff_dim = 1344
theta = 10000
eval_iters = 500
grad_clip = 1.0
save_iters = [500, 5000]

train_data = np.memmap(data_path, dtype = np.uint16, mode = 'r')
valid_data = np.memmap(valid_data_path, dtype = np.uint16, mode = 'r')

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT(vocab_size, context_len, embed_dim, n_head, n_layer, ff_dim, theta).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print(total_params)

val_loss = float('inf')
val_losses = []
val_iters = []

cuda
22696448


In [6]:
pbar = tqdm(range(max_iters))
for i in pbar:
    lr = get_lr(i, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    batch_x, batch_y = get_batch(train_data, context_len, batch_size, device)
    optimizer.zero_grad()

    with autocast(device_type = device.type, dtype = torch.bfloat16):
        logits = model(batch_x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if i in save_iters:
        save_path = f'storybot/model_iter_{i}.pt'
        model.save(save_path)
        print('모델이 저장되었습니다.')

    if (i % eval_iters) == 0 or i == max_iters -1:
        val_loss = evaluate(model, valid_data, context_len, batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(i)

    pbar.set_postfix({'loss': f'{loss.item():.4f}', 'val_loss': f'{val_loss:.4f}'})

model.save(model_save_path)

  5%|▌         | 500/10000 [01:22<17:24,  9.09it/s, loss=2.33, val_loss=9.29] 

모델이 저장되었습니다.


 50%|█████     | 5000/10000 [13:40<09:13,  9.04it/s, loss=1.69, val_loss=1.65]   

모델이 저장되었습니다.


100%|██████████| 10000/10000 [28:26<00:00,  5.86it/s, loss=1.45, val_loss=1.46]
